# XX - Test orchestration eurostat

## Importation des modules

In [ ]:
# Importation des modules
# Modules de base
import itertools
import sys
from pathlib import Path

import pandas as pd

# Racine du dépôt (le notebook est dans notebooks/) ajoutée au path pour importer
# le package macroforecast (importé depuis les sources, non installé).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Modules du package
from macroforecast.datasets import (
    EurostatClient,
    EurostatQueryRequestV30,
    StructureResourceType,
)
from macroforecast.datasets.sources.eurostat.parsing import parse_codelist_response
from macroforecast.datasets.core.download import download_updates, _schema_name
from macroforecast.datasets.utils import build_queries, filter_codes, load_split_filters

# Connecteur DuckLake (dt_ducklake_manager)
from dt_ducklake_manager import DuckLakeConnector

# Dataflow Comext étudié : commerce international (DS-045409)
DATAFLOW = "DS-045409"

# Instanciation du client Eurostat (API SDMX 3.0 par défaut)
client = EurostatClient()

## Téléchargement des données

### Requête de la structure

In [ ]:
# Requête de la structure du dataflow ds-045409 qui correspond aux données de commerce international
# On récupère et parse la DSD (Data Structure Definition) du dataflow.
#
# Comment déduit-on la codelist d'une dimension ? Dans la DSD SDMX, chaque dimension
# porte une balise LocalRepresentation/Enumeration qui référence — via une URN de la
# forme "...Codelist=ESTAT:CXT_FREE_ISO(10.0)" — la codelist énumérant ses valeurs
# autorisées. Le parser (parse_structure_response) extrait cet identifiant et l'expose
# dans l'attribut DimensionInfo.codelist : c'est ainsi que l'on sait que la dimension
# "reporter" est énumérée par CXT_FREE_ISO et "product" par CXT_NC.
structure = client.get_dataflow_structure(DATAFLOW)

# Tableau récapitulatif : la colonne "codelist" donne, pour chaque dimension, la
# codelist correspondante telle que lue dans la DSD.
structure_df = pd.DataFrame(
    [(d.position, d.name, d.description, d.codelist) for d in structure.dimensions],
    columns=["position", "name", "description", "codelist"],
).sort_values("position")
structure_df

### Requête des listes de codes

Les dimensions à scinder sont `reporter` et `product`. On déduit leur codelist
**directement de la structure** (colonne `codelist` ci-dessus, issue de la
`LocalRepresentation/Enumeration` de chaque dimension dans la DSD) plutôt que de
coder en dur les identifiants : `reporter` → `CXT_FREE_ISO`, `product` → `CXT_NC`.

Pour chaque codelist, la récupération se fait en **deux étapes** :
1. `client.get_structure(StructureResourceType.CODELIST, <id>)` → XML brut ;
2. `parse_codelist_response(xml)` → `DataFrame` `(code, name)`.

In [ ]:
# Requête des listes de codes
# Dimensions à scinder et déduction de leur codelist depuis la structure
REPORTER_DIM, PRODUCT_DIM = "reporter", "product"
dim_codelists = {d.name: d.codelist for d in structure.dimensions}
REPORTER_CODELIST = dim_codelists[REPORTER_DIM]   # -> CXT_FREE_ISO
PRODUCT_CODELIST = dim_codelists[PRODUCT_DIM]      # -> CXT_NC
print(f"reporter -> {REPORTER_CODELIST} | product -> {PRODUCT_CODELIST}")

# Liste des codes concernant le pays reporter
# Étape 1 : récupération du XML brut de la codelist (routé vers l'endpoint Comext)
reporter_xml = client.get_structure(StructureResourceType.CODELIST, REPORTER_CODELIST)
# Étape 2 : parsing du XML en DataFrame (code, name)
reporter_codes = parse_codelist_response(reporter_xml)
print(f"{len(reporter_codes)} codes reporter")
reporter_codes.head()

In [ ]:
# Liste des codes concernant le produit
# Même schéma en deux étapes, avec la codelist déduite de la structure (CXT_NC).
product_xml = client.get_structure(StructureResourceType.CODELIST, PRODUCT_CODELIST)
product_codes = parse_codelist_response(product_xml)
print(f"{len(product_codes)} codes produit")
product_codes.head()

### Construction des requêtes scindées

In [ ]:
# A partir des listes de codes précédentes on construit des requêtes indépendantes
# pour chaque pays reporter X code produit.
#
# Les codes itérés sont désormais pilotés par configuration : config/datasets/eurostat.yaml
# déclare, par dataflow et par dimension scindée, les filtres include/exclude (listes
# et motifs regex) appliqués aux codelists. Ici : reporter = 27 pays de l'UE ;
# product = codes SH2/SH4/SH6/NC8 valides (numériques de longueur 2/4/6/8), agrégats
# confidentiels "00*" exclus.
split_filters = load_split_filters(
    str(ROOT / "config" / "datasets" / "eurostat.yaml"), DATAFLOW
)
reporters = filter_codes(reporter_codes["code"], **split_filters["reporter"])
products = filter_codes(product_codes["code"], **split_filters["product"])
print(f"{len(reporters)} reporters | {len(products)} produits sélectionnés")

# Contrôle d'appartenance aux codelists récupérées précédemment
assert set(reporters) <= set(reporter_codes["code"]), "reporter inconnu de la codelist"
assert set(products) <= set(product_codes["code"]), "produit inconnu de la codelist"

# Dimensions communes (non scindées) à toutes les requêtes
# flow="*" : tous les flux (1=import, 2=export, 3=ré-export) — exports nécessaires
# au calcul de CDI3. indicators : masse ET valeur (la valeur sert aux CDI2/CDI3,
# définis sur les valeurs des échanges).
fixed_dims = {
    "freq": "A",                                          # Annuel
    "partner": "*",                                       # Tous les partenaires (pays + agrégats)
    "flow": ["1","2"],                                          # Les flux (import, export)
    "indicators": ["QUANTITY_IN_100KG", "VALUE_IN_EUROS"],  # Masse et valeur
}

# Produit cartésien reporter X produit → une requête indépendante par combinaison
all_queries = [
    EurostatQueryRequestV30(
        dataflow=DATAFLOW,
        dimensions={**fixed_dims, "reporter": reporter, "product": product},
    )
    for reporter, product in itertools.product(reporters, products)
]
print(f"{len(all_queries)} requêtes construites")

# A des fins de tests, on ne sélectionne que les deux premières requêtes de cette liste
# (le produit cartésien complet ~27k produits x 27 pays est très volumineux).
queries = all_queries[:2]
[q.identity_key() for q in queries]

### Requêtes de téléchargement initial

In [ ]:
# Téléchargement initial des deux requêtes (je te laisse choisir le nom des json et du catalogue ducklake)
# Emplacements de sortie (catalogue DuckLake + registres JSON du provider)
catalog_path = ROOT / "data" / "eurostat.ducklake"
data_path = ROOT / "data" / "eurostat"
structures_path = ROOT / "data" / "eurostat_structures.json"
last_download_path = ROOT / "data" / "eurostat_last_download.json"

# Création des répertoires de sortie si nécessaire
data_path.mkdir(parents=True, exist_ok=True)
structures_path.parent.mkdir(parents=True, exist_ok=True)

# Connecteur DuckLake (catalogue + stockage Parquet)
connector = DuckLakeConnector(str(catalog_path), str(data_path))

# Téléchargement initial : aucune date connue → récupération complète des séries
report = download_updates(
    client,
    queries,
    connector,
    structures_path=str(structures_path),
    last_download_path=str(last_download_path),
)
report

In [ ]:
# Vérification que les deux requêtes scindées sont dans la même table des faits du jeu de données.
# Le schéma DuckLake est dérivé du nom de dataflow (caractères non alphanumériques → "_").
schema = _schema_name(DATAFLOW)
print(f"Schéma DuckLake : {schema}")

conn = connector.connect()
try:
    # Clés primaires de la table : les dimensions du dataflow + TIME_PERIOD.
    # TIME_PERIOD est systématiquement ajouté aux clés primaires (cf.
    # download._primary_keys) pour garantir l'unicité d'une observation et éviter
    # les doublons lors des upserts incrémentaux.
    primary_keys = [
        r[0]
        for r in conn.execute(
            f"SELECT name FROM {schema}.metadata WHERE is_primary_key ORDER BY name"
        ).fetchall()
    ]
    print(f"Clés primaires : {primary_keys}")
    assert "TIME_PERIOD" in primary_keys, "TIME_PERIOD doit faire partie des clés primaires"

    # Absence de doublons : autant de lignes que de combinaisons de clé primaire
    n_rows = conn.execute(f"SELECT count(*) FROM {schema}.fact_table").fetchone()[0]
    n_unique = conn.execute(
        f"SELECT count(*) FROM "
        f"(SELECT 1 FROM {schema}.fact_table GROUP BY {', '.join(primary_keys)})"
    ).fetchone()[0]
    print(f"{n_rows} lignes / {n_unique} clés primaires distinctes (doublons : {n_rows - n_unique})")
    assert n_rows == n_unique, "présence de doublons sur la clé primaire"

    # Comptage par (reporter, product) : les deux séries scindées dans l'unique fact_table
    counts = conn.execute(
        f"SELECT reporter, product, count(*) AS n "
        f"FROM {schema}.fact_table "
        f"GROUP BY reporter, product ORDER BY reporter, product"
    ).fetch_df()
finally:
    conn.close()

counts

### Requêtes de mise à jour

In [ ]:
# Exécution des mêmes requêtes pour simuler une mise à jour
# Les dates de dernier téléchargement sont désormais connues : le client interroge
# d'abord la contrainte de données (dataconstraint) du dataflow et ne retélécharge
# que si les données ont été publiées depuis. Sans nouvelle publication, le rapport
# indique des requêtes "empty" (aucune nouvelle donnée écrite).
report_update = download_updates(
    client,
    queries,
    connector,
    structures_path=str(structures_path),
    last_download_path=str(last_download_path),
)
report_update

In [ ]:
# Fermeture propre des connexions HTTP du client
client.close()